# 🧊 Apache Iceberg — Marketplace de Máquinas Pesadas

## 📋 Cenário

Este notebook demonstra o uso do **Apache Iceberg** com **Apache Spark (PySpark)** no mesmo cenário
de **Marketplace de Máquinas Pesadas**, permitindo comparar as duas tecnologias lado a lado.

## 🎯 O que será demonstrado

1. Configuração da `SparkSession` com catálogo Iceberg (tipo Hadoop)
2. Criação de database e tabela com `USING iceberg`
3. Operações DML: `INSERT`, `SELECT`, `UPDATE`, `DELETE`
4. Consulta ao histórico de snapshots via `.snapshots`

## 📂 Nomenclatura Iceberg

No Iceberg, as tabelas seguem o padrão: `<catálogo>.<database>.<tabela>`

| Componente | Valor | Definido em |
|------------|-------|------------|
| Catálogo | `local` | `spark.sql.catalog.local` |
| Database | `trabalho` | `CREATE DATABASE local.trabalho` |
| Tabela | `maquinas_iceberg` | `CREATE TABLE` |
| Nome completo | `local.trabalho.maquinas_iceberg` | — |

## 🗃️ Estrutura da Tabela

| Campo | Tipo | Descrição |
|-------|------|----------|
| `id` | INT | Identificador único do anúncio |
| `tipo` | STRING | Tipo da máquina |
| `marca` | STRING | Fabricante |
| `modelo` | STRING | Modelo específico |
| `ano` | INT | Ano de fabricação |
| `preco` | DOUBLE | Preço de venda em R$ |
| `status` | STRING | Estado: `disponivel`, `em_negociacao`, `vendido` |

---

## ⚙️ 1. Configuração da SparkSession

Criamos uma `SparkSession` configurada com as extensões necessárias para o **Apache Iceberg**:

- **`iceberg-spark-runtime-3.5_2.12:1.6.1`** — runtime Iceberg para Spark 3.5 / Scala 2.12
- **`IcebergSparkSessionExtensions`** — habilita extensões SQL do Iceberg
- **`spark.sql.catalog.local`** — registra `local` como nome do catálogo Iceberg
- **`type: hadoop`** — catálogo baseado em sistema de arquivos (sem banco externo)

> ⏱️ **Nota:** Na **primeira execução**, o Spark fará o download dos JARs do Maven Central (~30–60s).
> As execuções seguintes serão mais rápidas pois os JARs ficam em cache em `~/.ivy2/`.

In [ ]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = (
    SparkSession
    .builder
    .appName("TrabalhoIceberg")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "spark-warehouse/iceberg")
    .getOrCreate()
)

print(f"✅ SparkSession iniciada com sucesso!")
print(f"   Versão do Spark  : {spark.version}")
print(f"   App Name         : {spark.sparkContext.appName}")
print(f"   Master           : {spark.sparkContext.master}")
print(f"   Spark UI         : http://localhost:4040")

---

## 🗄️ 2. Criação do Database e Tabela Iceberg

No Iceberg com catálogo Hadoop, precisamos criar o **database** (namespace) explicitamente antes
de criar a tabela. Em seguida, criamos a tabela com `USING iceberg`.

O Iceberg vai criar automaticamente:
- O diretório `spark-warehouse/iceberg/trabalho/maquinas_iceberg/`
- Os subdiretórios `data/` e `metadata/`
- O arquivo `metadata/v1.metadata.json` com o esquema inicial

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS local.trabalho")
print("✅ Database 'local.trabalho' criado (ou já existia)")

spark.sql("DROP TABLE IF EXISTS local.trabalho.maquinas_iceberg")
print("🗑️  Tabela anterior removida (se existia)")

spark.sql("""
    CREATE TABLE local.trabalho.maquinas_iceberg (
        id     INT,
        tipo   STRING,
        marca  STRING,
        modelo STRING,
        ano    INT,
        preco  DOUBLE,
        status STRING
    )
    USING iceberg
""")

print("✅ Tabela 'local.trabalho.maquinas_iceberg' criada com formato Iceberg!")

---

## ➕ 3. Inserção de Dados (INSERT)

Inserimos os mesmos 3 anúncios do notebook Delta Lake para facilitar a comparação.

Cada `INSERT` cria um novo **snapshot** no Iceberg (operação `append`).
O arquivo de metadados `metadata/v2.metadata.json` é criado apontando para o novo snapshot.

In [ ]:
spark.sql("""
    INSERT INTO local.trabalho.maquinas_iceberg VALUES
        (1, 'Escavadeira',      'Caterpillar', '320D', 2018, 350000.00, 'disponivel'),
        (2, 'Retroescavadeira', 'JCB',         '3CX',  2020, 280000.00, 'disponivel'),
        (3, 'Pa Carregadeira',  'Volvo',       'L90F', 2017, 420000.00, 'disponivel')
""")

print("✅ 3 máquinas inseridas com sucesso! (snapshot 'append' criado)")

---

## 🔍 4. Consulta Inicial (SELECT)

In [ ]:
print("📋 Estado inicial da tabela maquinas_iceberg:")
spark.sql(
    "SELECT * FROM local.trabalho.maquinas_iceberg ORDER BY id"
).show(truncate=False)

---

## ✏️ 5. Atualização de Dados (UPDATE)

A **JCB 3CX** (id=2) entrou em negociação. Atualizamos preço e status.

**Como o Iceberg executa o UPDATE internamente (Copy-on-Write):**

1. Lê o data file que contém o registro `id = 2`
2. Gera um **novo data file** com o registro atualizado
3. Cria um novo **snapshot** (`overwrite`) apontando para o novo data file
4. Atualiza o `metadata.json` com o novo snapshot

In [ ]:
spark.sql("""
    UPDATE local.trabalho.maquinas_iceberg
    SET preco  = 265000.00,
        status = 'em_negociacao'
    WHERE id = 2
""")

print("✅ Registro id=2 atualizado — JCB 3CX em negociação por R$ 265.000,00")
print("   Novo snapshot 'overwrite' criado")

In [ ]:
print("📋 Estado da tabela após UPDATE:")
spark.sql(
    "SELECT * FROM local.trabalho.maquinas_iceberg ORDER BY id"
).show(truncate=False)

---

## 🗑️ 6. Exclusão de Dados (DELETE)

A **Volvo L90F** (id=3) foi vendida. O anúncio será removido da plataforma.

O Iceberg cria um novo snapshot (`overwrite`) excluindo o registro, mantendo os data files
anteriores intactos para possibilitar Time Travel.

In [ ]:
spark.sql("""
    DELETE FROM local.trabalho.maquinas_iceberg
    WHERE id = 3
""")

print("✅ Registro id=3 removido — Volvo L90F foi vendida")
print("   Novo snapshot 'overwrite' criado")

In [ ]:
print("📋 Estado final da tabela após DELETE:")
spark.sql(
    "SELECT * FROM local.trabalho.maquinas_iceberg ORDER BY id"
).show(truncate=False)

---

## 📸 7. Histórico de Snapshots

O Iceberg mantém um histórico completo de **snapshots** — cada operação de escrita cria um
snapshot imutável. Podemos consultá-los diretamente via SQL usando a tabela virtual `.snapshots`:

| Coluna | Descrição |
|--------|----------|
| `committed_at` | Timestamp de quando o snapshot foi criado |
| `snapshot_id` | ID único do snapshot (usado para Time Travel) |
| `operation` | Tipo da operação: `append`, `overwrite`, `delete` |

### Time Travel com Iceberg

Use o `snapshot_id` obtido abaixo para consultar versões históricas:

```python
# Por snapshot_id
spark.sql("""
    SELECT * FROM local.trabalho.maquinas_iceberg
    VERSION AS OF <snapshot_id_aqui>
""").show()
```

In [ ]:
print("📸 Histórico de snapshots da tabela maquinas_iceberg:")
spark.sql("""
    SELECT committed_at, snapshot_id, operation
    FROM local.trabalho.maquinas_iceberg.snapshots
""").show(truncate=False)

---

## ✅ 8. Conclusão

Demonstramos com sucesso as principais funcionalidades do **Apache Iceberg** no cenário do Marketplace:

| Operação | Resultado |
|----------|----------|
| SparkSession com Iceberg | ✅ Configurada com catálogo Hadoop local |
| `CREATE DATABASE local.trabalho` | ✅ Namespace criado no catálogo |
| `CREATE TABLE ... USING iceberg` | ✅ Tabela com estrutura de metadados Iceberg |
| `INSERT INTO` | ✅ 3 anúncios inseridos (snapshot `append`) |
| `SELECT` | ✅ Dados consultados |
| `UPDATE` | ✅ JCB 3CX atualizada (snapshot `overwrite`) |
| `DELETE` | ✅ Volvo L90F removida (snapshot `overwrite`) |
| `.snapshots` | ✅ 3 snapshots de escrita exibidos |

## 🆚 Comparativo Final: Delta Lake vs. Apache Iceberg

| Aspecto | Delta Lake | Apache Iceberg |
|---------|-----------|----------------|
| Metadados | `_delta_log/` (JSON numerado) | `metadata/` (JSON + Avro em camadas) |
| Auditoria SQL | `DESCRIBE HISTORY tabela` | `SELECT ... FROM tabela.snapshots` |
| Nome do catálogo | `spark_catalog` (padrão) | Nome customizável (`local`) |
| Formato na criação | `USING delta` | `USING iceberg` |
| Operação INSERT | versão no log | snapshot `append` |
| Operação UPDATE/DELETE | versão no log | snapshot `overwrite` |

In [ ]:
spark.stop()
print("🛑 SparkSession encerrada com sucesso.")